# LAB 5 — Bài 1: Transformer cho bài toán phân loại văn bản

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook xây dựng một **Transformer Encoder từ đầu bằng PyTorch** để phân loại chủ đề câu tiếng Việt trên bộ dữ liệu UIT-VSFC.

## Mục tiêu học tập

Sau bài thực hành, sinh viên có thể:

1. Chuẩn bị dữ liệu văn bản cho bài toán phân loại nhiều lớp.
2. Giải thích vai trò của token `<cls>`, positional encoding và attention mask.
3. Cài đặt Multi-Head Self-Attention và Transformer Encoder.
4. Huấn luyện mô hình có validation, checkpoint, scheduler và early stopping.
5. Đánh giá bằng accuracy, macro precision, macro recall và macro F1.
6. Phân tích lỗi và quan sát attention của token `<cls>`.

## Quy trình tổng quát

```text
Câu đầu vào
    ↓
Token hóa + thêm <cls>
    ↓
Token Embedding + Positional Encoding
    ↓
N lớp Transformer Encoder
    ↓
Vector tại vị trí <cls>
    ↓
Linear Classifier
    ↓
Nhãn chủ đề
```

> **Lưu ý:** Đây là Transformer được cài đặt từ đầu để học nguyên lý, không phải mô hình pretrained như BERT hay PhoBERT.

## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng và huấn luyện Transformer.
- `scikit-learn`: tính metrics, classification report và confusion matrix.
- `matplotlib`: trực quan hóa dữ liệu, đường học và attention.

Chạy ô dưới đây một lần nếu môi trường chưa có đủ thư viện.

In [ ]:
%pip install -q scikit-learn matplotlib

## 2. Import, thiết lập seed và chọn thiết bị

### Vì sao cần random seed?

Khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp kết quả giữa các lần chạy ổn định hơn.

### CPU hay GPU?

PyTorch sẽ tự dùng CUDA nếu máy có GPU phù hợp. `pin_memory=True` và `non_blocking=True` chỉ thực sự có ích khi truyền dữ liệu lên GPU.

In [ ]:
from collections import Counter
from functools import partial
from pathlib import Path
from typing import Any
import json
import math
import os
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Hai thiết lập dưới đây ưu tiên khả năng tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "0") == "1"

print(f"Thiết bị: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")

## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB5_Bai1_Transformer_Classification.ipynb
UIT-VSFC/
├── UIT-VSFC-train.json
├── UIT-VSFC-dev.json
└── UIT-VSFC-test.json
```

Có thể đặt biến môi trường `VSFC_DIR` nếu dữ liệu nằm ở vị trí khác.

- **Train set:** cập nhật trọng số và xây vocabulary.
- **Dev set:** chọn checkpoint và dừng sớm.
- **Test set:** chỉ dùng một lần sau khi đã hoàn tất lựa chọn mô hình.

In [ ]:
DATA_DIR = Path(os.getenv("VSFC_DIR", "UIT-VSFC"))
TRAIN_PATH = DATA_DIR / "UIT-VSFC-train.json"
DEV_PATH = DATA_DIR / "UIT-VSFC-dev.json"
TEST_PATH = DATA_DIR / "UIT-VSFC-test.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "transformer_classification_best.pt"

required_paths = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    formatted_paths = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Không tìm thấy các tệp dữ liệu sau:\n"
        f"{formatted_paths}\n\n"
        "Hãy đặt thư mục UIT-VSFC cùng cấp với notebook "
        "hoặc thiết lập biến môi trường VSFC_DIR."
    )

print("Đã tìm thấy đầy đủ train, dev và test set.")
print(f"Checkpoint sẽ được lưu tại: {CHECKPOINT_PATH}")

## 4. Đọc và kiểm tra dữ liệu JSON

Mỗi phần tử dữ liệu cần có:

- `sentence`: câu đầu vào.
- `topic`: nhãn chủ đề.

Ta kiểm tra cấu trúc ngay khi đọc để lỗi dữ liệu xuất hiện sớm và có thông báo rõ ràng, thay vì phát sinh giữa lúc huấn luyện.

In [ ]:
def load_json_records(path: Path) -> list[dict[str, Any]]:
    """Đọc một tệp JSON và kiểm tra cấu trúc từng mẫu."""
    with path.open("r", encoding="utf-8") as file:
        records = json.load(file)

    if not isinstance(records, list):
        raise ValueError(f"Dữ liệu trong '{path}' phải là một danh sách.")

    for index, record in enumerate(records):
        if not isinstance(record, dict):
            raise TypeError(f"Mẫu {index} trong '{path}' phải là dictionary.")

        missing_fields = {"sentence", "topic"} - set(record)
        if missing_fields:
            raise ValueError(
                f"Mẫu {index} trong '{path}' thiếu trường: "
                f"{sorted(missing_fields)}"
            )

        if not isinstance(record["sentence"], str):
            raise TypeError(
                f"Trường sentence của mẫu {index} trong '{path}' phải là chuỗi."
            )

        if not record["sentence"].strip():
            raise ValueError(
                f"Trường sentence của mẫu {index} trong '{path}' đang rỗng."
            )

    return records


train_records = load_json_records(TRAIN_PATH)
dev_records = load_json_records(DEV_PATH)
test_records = load_json_records(TEST_PATH)

print(f"Train set: {len(train_records):,} mẫu")
print(f"Dev set:   {len(dev_records):,} mẫu")
print(f"Test set:  {len(test_records):,} mẫu")
print("\nVí dụ một mẫu train:")
print(train_records[0])

## 5. Khảo sát dữ liệu và kiểm tra nhãn

Không nên huấn luyện ngay sau khi đọc dữ liệu. Trước tiên cần xem:

- Số lượng mẫu ở từng split.
- Độ dài câu theo số token.
- Tập nhãn của dev/test có nằm trong tập nhãn train hay không.
- Có câu quá dài cần cắt bớt hay không.

Vocabulary và bảng nhãn **không được xây từ dev/test**, vì điều đó làm rò rỉ thông tin của tập đánh giá.

In [ ]:
def normalize_text(text: str) -> str:
    """Chuẩn hóa đơn giản cho mô hình word-level từ đầu."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    normalized = normalize_text(text)
    return normalized.split() if normalized else []


def summarize_split(name: str, records: list[dict[str, Any]]) -> dict[str, Any]:
    lengths = np.array([len(tokenize(item["sentence"])) for item in records])
    labels = {item["topic"] for item in records}

    return {
        "split": name,
        "samples": len(records),
        "labels": len(labels),
        "min_length": int(lengths.min()),
        "mean_length": float(lengths.mean()),
        "p95_length": float(np.percentile(lengths, 95)),
        "max_length": int(lengths.max()),
    }


for summary in [
    summarize_split("train", train_records),
    summarize_split("dev", dev_records),
    summarize_split("test", test_records),
]:
    print(summary)

train_label_set = {item["topic"] for item in train_records}
unknown_dev_labels = {item["topic"] for item in dev_records} - train_label_set
unknown_test_labels = {item["topic"] for item in test_records} - train_label_set

if unknown_dev_labels or unknown_test_labels:
    raise ValueError(
        "Dev/test chứa nhãn chưa xuất hiện trong train set. "
        f"Dev: {unknown_dev_labels}; Test: {unknown_test_labels}"
    )

train_lengths = [len(tokenize(item["sentence"])) for item in train_records]
plt.figure(figsize=(8, 4))
plt.hist(train_lengths, bins=30)
plt.xlabel("Số token trong câu")
plt.ylabel("Số mẫu")
plt.title("Phân bố độ dài câu trong train set")
plt.tight_layout()
plt.show()

## 6. Xây dựng vocabulary và bảng nhãn

### Ba token đặc biệt

- `<pad>`: lấp đầy các câu ngắn trong một batch.
- `<unk>`: thay thế từ không có trong vocabulary.
- `<cls>`: được thêm vào đầu mỗi câu; vector cuối cùng tại vị trí này dùng để phân loại.

Bản sinh viên lấy `x[:, 0]` nhưng không thêm token phân loại, vì vậy vị trí đầu chỉ là **từ đầu tiên của câu**. Bản này thêm `<cls>` một cách tường minh.

Vocabulary chỉ được xây từ **train set** và được sắp xếp cố định để ánh xạ token ổn định giữa các lần chạy.

In [ ]:
class Vocabulary:
    PAD_TOKEN = "<pad>"
    UNK_TOKEN = "<unk>"
    CLS_TOKEN = "<cls>"

    def __init__(
        self,
        sentences: list[str],
        min_frequency: int = 1,
        maximum_size: int | None = None,
    ) -> None:
        token_counts = Counter()
        for sentence in sentences:
            token_counts.update(tokenize(sentence))

        candidate_tokens = [
            token
            for token, frequency in token_counts.items()
            if frequency >= min_frequency
        ]
        candidate_tokens.sort(
            key=lambda token: (-token_counts[token], token)
        )

        if maximum_size is not None:
            number_of_normal_tokens = max(maximum_size - 3, 0)
            candidate_tokens = candidate_tokens[:number_of_normal_tokens]

        self.token_to_id = {
            self.PAD_TOKEN: 0,
            self.UNK_TOKEN: 1,
            self.CLS_TOKEN: 2,
        }
        for token in candidate_tokens:
            self.token_to_id[token] = len(self.token_to_id)

        self.id_to_token = {
            token_id: token
            for token, token_id in self.token_to_id.items()
        }
        self.pad_id = self.token_to_id[self.PAD_TOKEN]
        self.unk_id = self.token_to_id[self.UNK_TOKEN]
        self.cls_id = self.token_to_id[self.CLS_TOKEN]

    def __len__(self) -> int:
        return len(self.token_to_id)

    def encode(self, sentence: str, max_length: int) -> list[int]:
        """Thêm <cls>, đổi token thành ID và cắt chuỗi theo max_length."""
        if max_length < 1:
            raise ValueError("max_length phải lớn hơn hoặc bằng 1.")

        content_tokens = tokenize(sentence)[: max_length - 1]
        content_ids = [
            self.token_to_id.get(token, self.unk_id)
            for token in content_tokens
        ]
        return [self.cls_id, *content_ids]

    def decode(self, token_ids: list[int]) -> list[str]:
        return [
            self.id_to_token.get(token_id, self.UNK_TOKEN)
            for token_id in token_ids
        ]


MAX_LENGTH = 128
MIN_FREQUENCY = 1
MAX_VOCAB_SIZE = None

vocab = Vocabulary(
    sentences=[item["sentence"] for item in train_records],
    min_frequency=MIN_FREQUENCY,
    maximum_size=MAX_VOCAB_SIZE,
)

train_labels = sorted(train_label_set, key=str)
label_to_id = {
    label: label_id
    for label_id, label in enumerate(train_labels)
}
id_to_label = {
    label_id: label
    for label, label_id in label_to_id.items()
}

print(f"Kích thước vocabulary: {len(vocab):,}")
print(f"Số lớp: {len(label_to_id)}")
print(f"PAD_ID={vocab.pad_id}, UNK_ID={vocab.unk_id}, CLS_ID={vocab.cls_id}")
print(f"Ánh xạ nhãn: {label_to_id}")

## 7. Kiểm tra mã hóa một câu

Một bước kiểm tra nhỏ giúp phát hiện sớm các lỗi như:

- Quên thêm `<cls>`.
- ID đặc biệt bị trùng nhau.
- Chuỗi dài hơn `MAX_LENGTH`.
- Token không giải mã lại đúng.

Độ dài tối đa bao gồm cả token `<cls>`.

In [ ]:
sample_sentence = train_records[0]["sentence"]
sample_ids = vocab.encode(sample_sentence, max_length=MAX_LENGTH)

print(f"Câu gốc:     {sample_sentence}")
print(f"Token:       {tokenize(sample_sentence)}")
print(f"Token IDs:   {sample_ids}")
print(f"Giải mã:     {vocab.decode(sample_ids)}")
print(f"Nhãn:        {train_records[0]['topic']}")
print(f"Nhãn ID:     {label_to_id[train_records[0]['topic']]}")

assert sample_ids[0] == vocab.cls_id
assert len(sample_ids) <= MAX_LENGTH
assert len({vocab.pad_id, vocab.unk_id, vocab.cls_id}) == 3

## 8. Tạo `Dataset`

`Dataset` chịu trách nhiệm chuyển một mẫu thô thành:

- `input_ids`: chuỗi chỉ số token đã có `<cls>`.
- `label_id`: chỉ số lớp.
- `sentence`: câu gốc để phân tích lỗi về sau.

Không padding trong `__getitem__`; padding động sẽ được thực hiện theo batch để tránh lãng phí bộ nhớ.

In [ ]:
class VSFCClassificationDataset(Dataset):
    def __init__(
        self,
        records: list[dict[str, Any]],
        vocabulary: Vocabulary,
        label_mapping: dict[Any, int],
        max_length: int,
    ) -> None:
        self.records = records
        self.vocabulary = vocabulary
        self.label_mapping = label_mapping
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        input_ids = torch.tensor(
            self.vocabulary.encode(
                record["sentence"],
                max_length=self.max_length,
            ),
            dtype=torch.long,
        )
        label_id = torch.tensor(
            self.label_mapping[record["topic"]],
            dtype=torch.long,
        )

        return {
            "input_ids": input_ids,
            "label_id": label_id,
            "sentence": record["sentence"],
        }


train_dataset = VSFCClassificationDataset(
    train_records, vocab, label_to_id, MAX_LENGTH
)
dev_dataset = VSFCClassificationDataset(
    dev_records, vocab, label_to_id, MAX_LENGTH
)
test_dataset = VSFCClassificationDataset(
    test_records, vocab, label_to_id, MAX_LENGTH
)

sample = train_dataset[0]
print(f"input_ids shape: {tuple(sample['input_ids'].shape)}")
print(f"label_id: {sample['label_id'].item()}")
print(f"sentence: {sample['sentence']}")

## 9. Padding động và attention mask

Các câu trong batch có độ dài khác nhau. `collate_fn` sẽ:

1. Padding đến độ dài của câu dài nhất **trong batch**.
2. Tạo `attention_mask` với `True` tại token thật và `False` tại `<pad>`.
3. Ghép nhãn thành tensor một chiều.

Attention mask ngăn các token thật phân bổ trọng số vào vùng padding.

In [ ]:
def collate_batch(
    items: list[dict[str, Any]],
    pad_id: int,
) -> dict[str, Any]:
    sequences = [item["input_ids"] for item in items]
    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=pad_id,
    )
    attention_mask = input_ids.ne(pad_id)
    labels = torch.stack([item["label_id"] for item in items])
    lengths = attention_mask.sum(dim=1)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "lengths": lengths,
        "sentences": [item["sentence"] for item in items],
    }


batch_collator = partial(collate_batch, pad_id=vocab.pad_id)
example_batch = batch_collator(
    [train_dataset[index] for index in range(min(4, len(train_dataset)))]
)

print(f"input_ids:      {tuple(example_batch['input_ids'].shape)}")
print(f"attention_mask: {tuple(example_batch['attention_mask'].shape)}")
print(f"labels:         {tuple(example_batch['labels'].shape)}")
print(f"lengths:        {example_batch['lengths'].tolist()}")

assert example_batch["input_ids"].shape == example_batch["attention_mask"].shape
assert example_batch["attention_mask"].dtype == torch.bool

## 10. Tạo `DataLoader` và kiểm tra một batch

- Train loader dùng `shuffle=True` để thay đổi thứ tự mẫu ở mỗi epoch.
- Dev/test loader giữ nguyên thứ tự để đánh giá ổn định.
- `pin_memory=True` chỉ bật khi dùng CUDA.

Luôn kiểm tra shape của một batch trước khi xây mô hình.

In [ ]:
BATCH_SIZE = 16 if FAST_DEV_RUN else 32
PIN_MEMORY = DEVICE.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)

batch = next(iter(train_loader))
print(f"input_ids shape:      {tuple(batch['input_ids'].shape)}")
print(f"attention_mask shape: {tuple(batch['attention_mask'].shape)}")
print(f"labels shape:         {tuple(batch['labels'].shape)}")
print(f"Số batch train:       {len(train_loader)}")
print(f"5 độ dài đầu:         {batch['lengths'][:5].tolist()}")

## 11. Phân bố nhãn và tỷ lệ cắt chuỗi

Accuracy có thể gây hiểu nhầm khi dữ liệu lệch lớp. Vì vậy notebook theo dõi thêm **macro F1**, trong đó mỗi lớp có trọng số như nhau.

Ta cũng kiểm tra tỷ lệ câu bị cắt bởi `MAX_LENGTH`. Nếu tỷ lệ này cao, cần tăng giới hạn hoặc dùng phương pháp xử lý chuỗi dài phù hợp hơn.

In [ ]:
train_label_counts = Counter(item["topic"] for item in train_records)

print("Phân bố nhãn trong train set:")
for label in train_labels:
    print(f"- {label}: {train_label_counts[label]:,}")

plt.figure(figsize=(10, 5))
plt.bar(
    [str(label) for label in train_labels],
    [train_label_counts[label] for label in train_labels],
)
plt.xlabel("Nhãn chủ đề")
plt.ylabel("Số mẫu")
plt.title("Phân bố nhãn trong train set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

number_truncated = sum(
    len(tokenize(item["sentence"])) + 1 > MAX_LENGTH
    for item in train_records
)
truncation_rate = number_truncated / max(len(train_records), 1)
print(
    f"Số câu train bị cắt: {number_truncated:,} "
    f"({truncation_rate * 100:.2f}%)"
)

## 12. Sinusoidal Positional Encoding

Self-attention tự nó không biết token nào đứng trước hay sau. Positional encoding bổ sung thông tin vị trí vào embedding.

Với vị trí `pos` và chiều `i`:

\[
PE_{(pos, 2i)} = \sin\left(pos / 10000^{2i/d_{model}}\right)
\]

\[
PE_{(pos, 2i+1)} = \cos\left(pos / 10000^{2i/d_{model}}\right)
\]

Bảng này không cần học và được lưu bằng `register_buffer` để tự di chuyển theo thiết bị của mô hình.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model: int,
        dropout: float,
        max_length: int,
    ) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("encoding", encoding.unsqueeze(0))

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        sequence_length = x.size(1)
        if sequence_length > self.encoding.size(1):
            raise ValueError(
                f"Chuỗi dài {sequence_length}, vượt max_length "
                f"{self.encoding.size(1)} của positional encoding."
            )

        x = x + self.encoding[:, :sequence_length]
        x = self.dropout(x)
        return x * attention_mask.unsqueeze(-1).to(x.dtype)


position_demo = PositionalEncoding(
    d_model=16,
    dropout=0.0,
    max_length=MAX_LENGTH,
)
demo_input = torch.zeros(1, 20, 16)
demo_mask = torch.ones(1, 20, dtype=torch.bool)
demo_output = position_demo(demo_input, demo_mask)

plt.figure(figsize=(9, 4))
plt.imshow(demo_output[0].numpy(), aspect="auto")
plt.xlabel("Chiều embedding")
plt.ylabel("Vị trí token")
plt.title("Sinusoidal positional encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

## 13. Multi-Head Self-Attention

Từ cùng đầu vào `X`, mô hình chiếu thành Query, Key và Value:

\[
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Nhiều head cho phép mô hình quan sát các kiểu quan hệ khác nhau. Các điểm kỹ thuật cần chú ý:

- `d_model` phải chia hết cho số head.
- Mask được áp dụng lên **key padding** trước softmax.
- Dropout cần được áp dụng lên attention weights; bản sinh viên khai báo nhưng chưa dùng.
- Khi cần phân tích, lớp trả thêm ma trận attention.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError("d_model phải chia hết cho num_heads.")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dimension = d_model // num_heads

        self.qkv_projection = nn.Linear(d_model, 3 * d_model)
        self.output_projection = nn.Linear(d_model, d_model)
        self.attention_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, sequence_length, _ = x.shape

        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dimension,
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(dim=0)

        scores = torch.matmul(queries, keys.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dimension)

        key_mask = attention_mask[:, None, None, :]
        scores = scores.masked_fill(
            ~key_mask,
            torch.finfo(scores.dtype).min,
        )

        attention_weights = torch.softmax(scores, dim=-1)
        dropped_attention = self.attention_dropout(attention_weights)

        context = torch.matmul(dropped_attention, values)
        context = context.transpose(1, 2).contiguous().reshape(
            batch_size,
            sequence_length,
            self.d_model,
        )
        output = self.output_projection(context)

        return output, attention_weights


attention_demo = MultiHeadSelfAttention(
    d_model=32,
    num_heads=4,
    dropout=0.1,
)
demo_x = torch.randn(2, 7, 32)
demo_attention_mask = torch.tensor(
    [
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1],
    ],
    dtype=torch.bool,
)
demo_attention_output, demo_attention_weights = attention_demo(
    demo_x,
    demo_attention_mask,
)

print(f"Attention output:  {tuple(demo_attention_output.shape)}")
print(f"Attention weights: {tuple(demo_attention_weights.shape)}")
assert demo_attention_weights[0, :, :, 5:].max().item() < 1e-6

## 14. Transformer Encoder Layer

Mỗi encoder layer gồm hai khối:

1. Multi-Head Self-Attention.
2. Position-wise Feed-Forward Network.

Mỗi khối có residual connection, LayerNorm và Dropout. Bản này dùng **Pre-LayerNorm**:

```text
x → LayerNorm → Attention → Dropout → cộng residual
x → LayerNorm → Feed Forward → Dropout → cộng residual
```

Pre-LayerNorm thường giúp quá trình tối ưu ổn định hơn khi tự huấn luyện Transformer.

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.self_attention = MultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.attention_norm = nn.LayerNorm(d_model)
        self.feed_forward_norm = nn.LayerNorm(d_model)
        self.residual_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        normalized_x = self.attention_norm(x)
        attention_output, attention_weights = self.self_attention(
            normalized_x,
            attention_mask,
        )
        x = x + self.residual_dropout(attention_output)

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(x)
        )
        x = x + self.residual_dropout(feed_forward_output)

        # Không để biểu diễn tại vị trí padding tiếp tục lan truyền qua các tầng.
        x = x * attention_mask.unsqueeze(-1).to(x.dtype)
        return x, attention_weights


class TransformerEncoder(nn.Module):
    def __init__(
        self,
        num_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                TransformerEncoderLayer(
                    d_model=d_model,
                    num_heads=num_heads,
                    d_ff=d_ff,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        final_attention_weights = None

        for layer in self.layers:
            x, final_attention_weights = layer(x, attention_mask)

        x = self.final_norm(x)
        x = x * attention_mask.unsqueeze(-1).to(x.dtype)
        return x, final_attention_weights

## 15. Mô hình Transformer Classifier

Quy trình forward:

1. Chuyển token ID thành embedding.
2. Nhân embedding với `√d_model`.
3. Cộng positional encoding.
4. Đưa qua các encoder layer.
5. Lấy biểu diễn tại vị trí `<cls>`.
6. Đưa qua lớp tuyến tính để tạo logits cho các lớp.

Logits chưa qua softmax vì `CrossEntropyLoss` đã thực hiện phép biến đổi cần thiết một cách ổn định số học.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        number_of_classes: int,
        pad_id: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        dropout: float,
        max_length: int,
    ) -> None:
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.embedding = nn.Embedding(
            vocabulary_size,
            d_model,
            padding_idx=pad_id,
        )
        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            dropout=dropout,
            max_length=max_length,
        )
        self.encoder = TransformerEncoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, number_of_classes),
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        if attention_mask is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.positional_encoding(x, attention_mask)
        encoded, final_attention_weights = self.encoder(
            x,
            attention_mask,
        )

        cls_representation = encoded[:, 0]
        logits = self.classifier(cls_representation)

        if return_attention:
            return logits, final_attention_weights
        return logits

## 16. Khởi tạo mô hình và kiểm tra forward pass

Kiểm tra forward pass trước khi huấn luyện giúp xác nhận:

- Shape logits là `[batch_size, number_of_classes]`.
- Attention có shape `[batch_size, heads, query_length, key_length]`.
- Cấu hình không vi phạm điều kiện `d_model % num_heads == 0`.

`FAST_DEV_RUN=1` dùng mô hình nhỏ để kiểm tra pipeline, không phải cấu hình báo cáo kết quả cuối cùng.

In [ ]:
MODEL_CONFIG = {
    "pad_id": vocab.pad_id,
    "d_model": 32 if FAST_DEV_RUN else 128,
    "num_heads": 4,
    "num_layers": 1 if FAST_DEV_RUN else 3,
    "d_ff": 64 if FAST_DEV_RUN else 256,
    "dropout": 0.15,
    "max_length": MAX_LENGTH,
}

model = TransformerClassifier(
    vocabulary_size=len(vocab),
    number_of_classes=len(label_to_id),
    **MODEL_CONFIG,
).to(DEVICE)


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


batch_on_device = {
    key: value.to(DEVICE)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

with torch.no_grad():
    logits, attention_weights = model(
        batch_on_device["input_ids"],
        batch_on_device["attention_mask"],
        return_attention=True,
    )

print(model)
print(f"\nSố tham số có thể học: {count_trainable_parameters(model):,}")
print(f"Logits shape:    {tuple(logits.shape)}")
print(f"Attention shape: {tuple(attention_weights.shape)}")

assert logits.shape == (batch["input_ids"].size(0), len(label_to_id))
assert attention_weights.shape[-1] == batch["input_ids"].size(1)

## 17. Loss, optimizer và learning-rate scheduler

- `CrossEntropyLoss`: phù hợp với phân loại nhiều lớp loại trừ nhau.
- `AdamW`: tách weight decay khỏi bước cập nhật gradient.
- `ReduceLROnPlateau`: giảm learning rate khi dev macro F1 không cải thiện.
- Gradient clipping: hạn chế gradient quá lớn.

Checkpoint được chọn theo **dev macro F1**, không theo test set.

In [ ]:
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0
EPOCHS = 2 if FAST_DEV_RUN else 25
PATIENCE = 2 if FAST_DEV_RUN else 5

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

print(f"Loss: {loss_function}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Số epoch tối đa: {EPOCHS}")

## 18. Hàm huấn luyện một epoch

Trong mỗi batch:

1. Chuyển tensor sang thiết bị.
2. Xóa gradient cũ.
3. Forward để tính logits.
4. Tính loss và backward.
5. Clip gradient.
6. Cập nhật trọng số.

Loss trung bình được tính theo **số mẫu**, nên vẫn đúng khi batch cuối nhỏ hơn các batch trước.

In [ ]:
def move_batch_to_device(
    batch: dict[str, Any],
    device: torch.device,
) -> dict[str, Any]:
    return {
        key: value.to(
            device,
            non_blocking=device.type == "cuda",
        )
        if isinstance(value, torch.Tensor)
        else value
        for key, value in batch.items()
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function: nn.Module,
    device: torch.device,
    max_grad_norm: float,
) -> float:
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
        )
        loss = loss_function(logits, batch["labels"])

        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_grad_norm,
        )
        optimizer.step()

        batch_size = batch["labels"].size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / max(total_samples, 1)

## 19. Hàm đánh giá

Khi đánh giá:

- Dùng `model.eval()` để tắt Dropout.
- Dùng `torch.inference_mode()` để không lưu đồ thị gradient.
- Thu thập dự đoán của toàn bộ tập trước khi tính metrics.

Macro precision, recall và F1 cho mỗi lớp trọng số bằng nhau, phù hợp hơn accuracy khi phân bố lớp không cân bằng.

In [ ]:
@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> dict[str, Any]:
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels: list[int] = []
    all_predictions: list[int] = []
    all_probabilities: list[np.ndarray] = []
    all_sentences: list[str] = []

    for batch in loader:
        raw_sentences = batch["sentences"]
        batch = move_batch_to_device(batch, device)

        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
        )
        loss = loss_function(logits, batch["labels"])
        probabilities = torch.softmax(logits, dim=-1)
        predictions = probabilities.argmax(dim=-1)

        batch_size = batch["labels"].size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.extend(batch["labels"].cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().numpy())
        all_sentences.extend(raw_sentences)

    labels_array = np.asarray(all_labels)
    predictions_array = np.asarray(all_predictions)
    probabilities_array = np.asarray(all_probabilities)

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": accuracy_score(labels_array, predictions_array),
        "precision": precision_score(
            labels_array,
            predictions_array,
            average="macro",
            zero_division=0,
        ),
        "recall": recall_score(
            labels_array,
            predictions_array,
            average="macro",
            zero_division=0,
        ),
        "f1": f1_score(
            labels_array,
            predictions_array,
            average="macro",
            zero_division=0,
        ),
        "labels": labels_array,
        "predictions": predictions_array,
        "probabilities": probabilities_array,
        "sentences": all_sentences,
    }

## 20. Vòng lặp huấn luyện, checkpoint và early stopping

Sau mỗi epoch:

1. Huấn luyện trên train set.
2. Đánh giá trên dev set.
3. Điều chỉnh learning rate từ dev macro F1.
4. Lưu checkpoint nếu dev macro F1 tốt hơn.
5. Dừng sớm nếu không cải thiện sau `PATIENCE` epoch.

Checkpoint lưu cả trọng số và các ánh xạ cần thiết để tái tạo mô hình.

In [ ]:
history = {
    "train_loss": [],
    "dev_loss": [],
    "dev_accuracy": [],
    "dev_f1": [],
    "learning_rate": [],
}

best_dev_f1 = -math.inf
epochs_without_improvement = 0
training_start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        loss_function=loss_function,
        device=DEVICE,
        max_grad_norm=MAX_GRAD_NORM,
    )
    dev_metrics = evaluate_model(
        model=model,
        loader=dev_loader,
        loss_function=loss_function,
        device=DEVICE,
    )

    current_learning_rate = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_metrics["loss"])
    history["dev_accuracy"].append(dev_metrics["accuracy"])
    history["dev_f1"].append(dev_metrics["f1"])
    history["learning_rate"].append(current_learning_rate)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Dev loss: {dev_metrics['loss']:.4f} | "
        f"Dev accuracy: {dev_metrics['accuracy'] * 100:.2f}% | "
        f"Dev macro F1: {dev_metrics['f1'] * 100:.2f}% | "
        f"LR: {current_learning_rate:.2e}"
    )

    scheduler.step(dev_metrics["f1"])

    if dev_metrics["f1"] > best_dev_f1:
        best_dev_f1 = dev_metrics["f1"]
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "best_dev_f1": best_dev_f1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "model_config": MODEL_CONFIG,
                "token_to_id": vocab.token_to_id,
                "label_to_id": label_to_id,
                "id_to_label": id_to_label,
                "max_length": MAX_LENGTH,
                "normalization": "lowercase + remove punctuation + whitespace",
            },
            CHECKPOINT_PATH,
        )
        print("  ↳ Đã lưu checkpoint tốt nhất.")
    else:
        epochs_without_improvement += 1
        print(
            "  ↳ Dev macro F1 chưa cải thiện: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Dừng sớm vì dev macro F1 không cải thiện "
            f"trong {PATIENCE} epoch liên tiếp."
        )
        break

training_time = time.time() - training_start_time
print(f"\nThời gian huấn luyện: {training_time:.1f} giây")
print(f"Dev macro F1 tốt nhất: {best_dev_f1 * 100:.2f}%")

## 21. Trực quan hóa quá trình học

Đồ thị giúp phát hiện:

- **Underfitting:** train và dev loss đều cao.
- **Overfitting:** train loss tiếp tục giảm nhưng dev loss tăng hoặc dev F1 giảm.
- Learning rate đã được scheduler giảm ở thời điểm nào.

Không nên chỉ nhìn train loss để kết luận mô hình tốt.

In [ ]:
epoch_numbers = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epoch_numbers, history["train_loss"], marker="o", label="Train loss")
plt.plot(epoch_numbers, history["dev_loss"], marker="o", label="Dev loss")
plt.xlabel("Epoch")
plt.ylabel("Loss trung bình trên mẫu")
plt.title("Loss theo epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(
    epoch_numbers,
    np.array(history["dev_accuracy"]) * 100,
    marker="o",
    label="Dev accuracy",
)
plt.plot(
    epoch_numbers,
    np.array(history["dev_f1"]) * 100,
    marker="o",
    label="Dev macro F1",
)
plt.xlabel("Epoch")
plt.ylabel("Điểm số (%)")
plt.title("Chỉ số trên dev set")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(epoch_numbers, history["learning_rate"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning rate theo epoch")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 22. Tải checkpoint tốt nhất, đánh giá test và phân tích lỗi

Ta tải lại checkpoint tốt nhất thay vì dùng trạng thái của epoch cuối. Sau đó:

- Báo cáo loss, accuracy và macro metrics.
- In classification report theo từng lớp.
- Vẽ confusion matrix.
- Xem các câu dự đoán sai có độ tin cậy cao.
- Quan sát attention trung bình từ `<cls>` đến các token của một câu.

Attention chỉ là một tín hiệu phân tích; không nên mặc định xem nó là lời giải thích nhân quả hoàn chỉnh.

In [ ]:
def load_checkpoint(path: Path, device: torch.device) -> dict[str, Any]:
    """Tương thích cả phiên bản PyTorch có và chưa có weights_only."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


best_checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)
best_model = TransformerClassifier(
    vocabulary_size=len(best_checkpoint["token_to_id"]),
    number_of_classes=len(best_checkpoint["label_to_id"]),
    **best_checkpoint["model_config"],
).to(DEVICE)
best_model.load_state_dict(best_checkpoint["model_state_dict"])

print(
    f"Checkpoint epoch {best_checkpoint['epoch']} | "
    f"Dev macro F1: {best_checkpoint['best_dev_f1'] * 100:.2f}%"
)

test_metrics = evaluate_model(
    model=best_model,
    loader=test_loader,
    loss_function=loss_function,
    device=DEVICE,
)

ordered_label_ids = np.arange(len(id_to_label))
ordered_label_names = [str(id_to_label[index]) for index in ordered_label_ids]

print("\nKẾT QUẢ TRÊN TEST SET")
print("-" * 48)
print(f"Test loss:       {test_metrics['loss']:.4f}")
print(f"Test accuracy:   {test_metrics['accuracy'] * 100:.2f}%")
print(f"Macro precision: {test_metrics['precision'] * 100:.2f}%")
print(f"Macro recall:    {test_metrics['recall'] * 100:.2f}%")
print(f"Macro F1-score:  {test_metrics['f1'] * 100:.2f}%")

print("\nClassification report:\n")
print(
    classification_report(
        test_metrics["labels"],
        test_metrics["predictions"],
        labels=ordered_label_ids,
        target_names=ordered_label_names,
        digits=4,
        zero_division=0,
    )
)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay.from_predictions(
    test_metrics["labels"],
    test_metrics["predictions"],
    labels=ordered_label_ids,
    display_labels=ordered_label_names,
    values_format="d",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion matrix trên test set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

wrong_indices = np.where(
    test_metrics["labels"] != test_metrics["predictions"]
)[0]
wrong_indices = sorted(
    wrong_indices,
    key=lambda index: test_metrics["probabilities"][index].max(),
    reverse=True,
)

print("\nMột số mẫu sai có độ tin cậy cao:")
for rank, index in enumerate(wrong_indices[:10], start=1):
    true_id = int(test_metrics["labels"][index])
    predicted_id = int(test_metrics["predictions"][index])
    confidence = float(test_metrics["probabilities"][index, predicted_id])
    print(f"\n[{rank}] {test_metrics['sentences'][index]}")
    print(f"    Nhãn thật: {id_to_label[true_id]}")
    print(f"    Dự đoán:   {id_to_label[predicted_id]}")
    print(f"    Tin cậy:   {confidence * 100:.2f}%")


@torch.inference_mode()
def visualize_cls_attention(
    model: nn.Module,
    sentence: str,
    vocabulary: Vocabulary,
    device: torch.device,
    max_length: int,
) -> None:
    token_ids = vocabulary.encode(sentence, max_length=max_length)
    tokens = vocabulary.decode(token_ids)
    input_ids = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)
    attention_mask = input_ids.ne(vocabulary.pad_id)

    model.eval()
    _, attention_weights = model(
        input_ids,
        attention_mask,
        return_attention=True,
    )

    # Trung bình các head; query ở vị trí 0 chính là <cls>.
    cls_attention = attention_weights[0, :, 0, :].mean(dim=0).cpu().numpy()

    plt.figure(figsize=(max(8, len(tokens) * 0.6), 3))
    plt.bar(range(len(tokens)), cls_attention)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.ylabel("Attention weight")
    plt.title("Attention trung bình từ <cls> ở encoder layer cuối")
    plt.tight_layout()
    plt.show()


visualize_cls_attention(
    model=best_model,
    sentence=test_records[0]["sentence"],
    vocabulary=vocab,
    device=DEVICE,
    max_length=MAX_LENGTH,
)

## 23. Dự đoán câu mới, tổng kết và bài tập mở rộng

Hàm dưới đây trả về:

- Nhãn dự đoán.
- Độ tin cậy cao nhất.
- Phân phối xác suất của tất cả lớp.

### Tổng kết

Notebook đã hoàn thiện pipeline:

```text
Dữ liệu → Vocabulary train-only → <cls> + Padding/Mask
→ Transformer Encoder → Dev macro F1
→ Checkpoint tốt nhất → Test → Phân tích lỗi
```

### Hạn chế

- Tokenization theo khoảng trắng còn đơn giản đối với tiếng Việt.
- Embedding và Transformer đều học từ đầu nên cần nhiều dữ liệu.
- Chuỗi dài hơn `MAX_LENGTH` bị cắt.
- Greedy class prediction chưa biểu diễn độ bất định đầy đủ.

### Bài tập mở rộng

1. So sánh `mean pooling` với `<cls>` pooling.
2. Thử `num_layers`, `num_heads`, `d_model` và dropout khác nhau.
3. Dùng class weight hoặc focal loss khi dữ liệu lệch lớp.
4. Thay word-level tokenization bằng subword tokenizer.
5. So sánh với PhoBERT và báo cáo accuracy, macro F1, tốc độ, số tham số.

In [ ]:
@torch.inference_mode()
def predict_topic(
    model: nn.Module,
    sentence: str,
    vocabulary: Vocabulary,
    id_to_label: dict[int, Any],
    device: torch.device,
    max_length: int,
) -> dict[str, Any]:
    if not sentence.strip():
        raise ValueError("Câu đầu vào không được rỗng.")

    token_ids = vocabulary.encode(sentence, max_length=max_length)
    input_ids = torch.tensor(
        token_ids,
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)
    attention_mask = input_ids.ne(vocabulary.pad_id)

    model.eval()
    logits = model(input_ids, attention_mask)
    probabilities = torch.softmax(logits, dim=-1)[0]
    confidence, predicted_id = probabilities.max(dim=0)

    class_probabilities = {
        id_to_label[class_id]: float(probabilities[class_id].item())
        for class_id in range(len(id_to_label))
    }
    class_probabilities = dict(
        sorted(
            class_probabilities.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    )

    return {
        "sentence": sentence,
        "predicted_label": id_to_label[int(predicted_id.item())],
        "confidence": float(confidence.item()),
        "class_probabilities": class_probabilities,
        "tokens": vocabulary.decode(token_ids),
        "was_truncated": len(tokenize(sentence)) + 1 > max_length,
    }


example_sentence = "Giảng viên giảng bài rất dễ hiểu và nhiệt tình."
prediction = predict_topic(
    model=best_model,
    sentence=example_sentence,
    vocabulary=vocab,
    id_to_label=id_to_label,
    device=DEVICE,
    max_length=MAX_LENGTH,
)

print(f"Câu: {prediction['sentence']}")
print(f"Token mô hình nhận: {prediction['tokens']}")
print(f"Chủ đề dự đoán: {prediction['predicted_label']}")
print(f"Độ tin cậy: {prediction['confidence'] * 100:.2f}%")
print("\nXác suất theo lớp:")
for label, probability in prediction["class_probabilities"].items():
    print(f"- {label}: {probability * 100:.2f}%")